# Préparer les données pour Label Studio

Objectif :

- partir de la sortie OCR Tesseract ;
- conserver les mots et leurs bounding boxes ;
- convertir les coordonnées dans le format attendu par Label Studio ;
- générer un premier fichier JSON importable ;
- valider le processus sur un devis avant de l’automatiser.

In [1]:
import json
from pathlib import Path

import pymupdf
import pytesseract

from PIL import Image
from pytesseract import Output

In [3]:
pdf_path = Path("../benchmarks/ocr/input/devis_2.pdf")

document = pymupdf.open(pdf_path)
page = document[0]

pix = page.get_pixmap(dpi=200, alpha=False)

image = Image.frombytes(
    "RGB",
    [pix.width, pix.height],
    pix.samples,
)

image_width, image_height = image.size

print(image_width, image_height)

1654 2339


In [4]:
ocr_data = pytesseract.image_to_data(
    image,
    lang="fra",
    output_type=Output.DICT,
    config="--oem 1 --psm 3",
)

print(len(ocr_data["text"]))

347


In [5]:
words = []

for index, raw_text in enumerate(ocr_data["text"]):
    text = raw_text.strip()
    confidence = float(ocr_data["conf"][index])

    if not text or confidence < 0:
        continue

    left = int(ocr_data["left"][index])
    top = int(ocr_data["top"][index])
    width = int(ocr_data["width"][index])
    height = int(ocr_data["height"][index])

    words.append(
        {
            "text": text,
            "confidence": round(confidence / 100, 4),
            "bbox_pixels": [
                left,
                top,
                left + width,
                top + height,
            ],
        }
    )

print(f"{len(words)} mots détectés")
words[:10]

181 mots détectés


[{'text': 'Cäblage', 'confidence': 0.5, 'bbox_pixels': [164, 214, 200, 229]},
 {'text': 'Maintens', 'confidence': 0.41, 'bbox_pixels': [211, 214, 253, 229]},
 {'text': 'Offre', 'confidence': 0.95, 'bbox_pixels': [103, 366, 197, 400]},
 {'text': 'de', 'confidence': 0.95, 'bbox_pixels': [210, 366, 252, 394]},
 {'text': 'prix', 'confidence': 0.96, 'bbox_pixels': [260, 366, 343, 401]},
 {'text': 'n°', 'confidence': 0.85, 'bbox_pixels': [356, 368, 415, 395]},
 {'text': ':', 'confidence': 0.85, 'bbox_pixels': [403, 362, 418, 405]},
 {'text': '002.143/25',
  'confidence': 0.91,
  'bbox_pixels': [429, 366, 616, 395]},
 {'text': 'Affaire', 'confidence': 0.93, 'bbox_pixels': [97, 414, 173, 437]},
 {'text': ':', 'confidence': 0.91, 'bbox_pixels': [180, 421, 186, 433]}]

In [6]:
def bbox_to_percentages(
    bbox: list[int],
    image_width: int,
    image_height: int,
) -> dict[str, float]:
    x1, y1, x2, y2 = bbox

    return {
        "x": 100 * x1 / image_width,
        "y": 100 * y1 / image_height,
        "width": 100 * (x2 - x1) / image_width,
        "height": 100 * (y2 - y1) / image_height,
    }

In [7]:
first_word = words[0]

bbox_percent = bbox_to_percentages(
    first_word["bbox_pixels"],
    image_width,
    image_height,
)

print(first_word)
print(bbox_percent)

{'text': 'Cäblage', 'confidence': 0.5, 'bbox_pixels': [164, 214, 200, 229]}
{'x': 9.915356711003627, 'y': 9.149209063702436, 'width': 2.176541717049577, 'height': 0.6412997007268063}


In [8]:
regions = []

for index, word in enumerate(words):
    position = bbox_to_percentages(
        word["bbox_pixels"],
        image_width,
        image_height,
    )

    region_id = f"word-{index}"

    regions.append(
        {
            "id": region_id,
            "from_name": "bbox",
            "to_name": "image",
            "type": "rectangle",
            "value": {
                "x": position["x"],
                "y": position["y"],
                "width": position["width"],
                "height": position["height"],
                "rotation": 0,
            },
            "original_width": image_width,
            "original_height": image_height,
            "image_rotation": 0,
        }
    )

    regions.append(
        {
            "id": region_id,
            "from_name": "transcription",
            "to_name": "image",
            "type": "textarea",
            "value": {
                "x": position["x"],
                "y": position["y"],
                "width": position["width"],
                "height": position["height"],
                "rotation": 0,
                "text": [word["text"]],
            },
            "original_width": image_width,
            "original_height": image_height,
            "image_rotation": 0,
        }
    )

print(f"{len(words)} mots")
print(f"{len(regions)} régions Label Studio")
regions[:2]

181 mots
362 régions Label Studio


[{'id': 'word-0',
  'from_name': 'bbox',
  'to_name': 'image',
  'type': 'rectangle',
  'value': {'x': 9.915356711003627,
   'y': 9.149209063702436,
   'width': 2.176541717049577,
   'height': 0.6412997007268063,
   'rotation': 0},
  'original_width': 1654,
  'original_height': 2339,
  'image_rotation': 0},
 {'id': 'word-0',
  'from_name': 'transcription',
  'to_name': 'image',
  'type': 'textarea',
  'value': {'x': 9.915356711003627,
   'y': 9.149209063702436,
   'width': 2.176541717049577,
   'height': 0.6412997007268063,
   'rotation': 0,
   'text': ['Cäblage']},
  'original_width': 1654,
  'original_height': 2339,
  'image_rotation': 0}]

In [9]:
output_image_dir = Path("./data/label_studio/images")
output_image_dir.mkdir(parents=True, exist_ok=True)

image_path = output_image_dir / "devis_2_page_1.png"

image.save(image_path)

print(image_path)

data/label_studio/images/devis_2_page_1.png


In [15]:
task = {
    "data": {
         "image": "/data/local-files/?d=data/label_studio/images/devis_2_page_1.png"
    },
    "predictions": [
        {
            "model_version": "tesseract-v1",
            "score": 1.0,
            "result": regions,
        }
    ],
}

task

{'data': {'image': '/data/local-files/?d=data/label_studio/images/devis_2_page_1.png'},
 'predictions': [{'model_version': 'tesseract-v1',
   'score': 1.0,
   'result': [{'id': 'word-0',
     'from_name': 'bbox',
     'to_name': 'image',
     'type': 'rectangle',
     'value': {'x': 9.915356711003627,
      'y': 9.149209063702436,
      'width': 2.176541717049577,
      'height': 0.6412997007268063,
      'rotation': 0},
     'original_width': 1654,
     'original_height': 2339,
     'image_rotation': 0},
    {'id': 'word-0',
     'from_name': 'transcription',
     'to_name': 'image',
     'type': 'textarea',
     'value': {'x': 9.915356711003627,
      'y': 9.149209063702436,
      'width': 2.176541717049577,
      'height': 0.6412997007268063,
      'rotation': 0,
      'text': ['Cäblage']},
     'original_width': 1654,
     'original_height': 2339,
     'image_rotation': 0},
    {'id': 'word-1',
     'from_name': 'bbox',
     'to_name': 'image',
     'type': 'rectangle',
     'value

In [16]:
output_json_dir = Path("./data/label_studio/tasks")
output_json_dir.mkdir(parents=True, exist_ok=True)

task_path = output_json_dir / "devis_2_page_1.json"

with task_path.open("w", encoding="utf-8") as file:
    json.dump(
        [task],
        file,
        ensure_ascii=False,
        indent=2,
    )

print(task_path)

data/label_studio/tasks/devis_2_page_1.json


Je pense que c'est une excellente idée. Ce résumé doit être suffisamment complet pour que tu puisses **reprendre dans plusieurs semaines** sans repartir de zéro et qu'il puisse également te servir de support pour la soutenance.

---

# Notebook 02 — Comprendre l'entrée de LayoutLMv3

## Objectif

L'objectif de ce notebook n'était **pas** d'entraîner LayoutLMv3 ni de réaliser des annotations.

L'objectif était de répondre à une seule question :

> **Que reçoit exactement LayoutLMv3 en entrée ?**

Cette compréhension est essentielle avant de commencer les annotations ou le fine-tuning.

---

# Position du notebook dans le pipeline

```text
Benchmark OCR
        │
        ▼
Comprendre l'entrée de LayoutLMv3   ← Notebook 02
        │
        ▼
Préparer les données Label Studio
        │
        ▼
Annotations
        │
        ▼
Préparation du dataset
        │
        ▼
Fine-tuning LayoutLMv3
```

---

# Ce que nous avons appris

## 1. LayoutLMv3 ne lit pas directement un PDF

Un PDF est un conteneur.

Le modèle ne sait pas interpréter directement un PDF.

Nous devons d'abord transformer le PDF en image.

Pour cela, nous utilisons **PyMuPDF**.

```text
PDF
        │
        ▼
PyMuPDF
        │
        ▼
Image
```

Nous avons choisi PyMuPDF car :

* rapide
* excellente qualité de rendu
* contrôle du DPI
* largement utilisé dans les projets Document AI

---

## 2. Pourquoi un OCR est nécessaire

LayoutLMv3 ne sait pas lire les caractères présents dans une image.

Nous avons donc besoin d'un OCR.

L'OCR fournit :

* le texte reconnu
* la position de chaque mot
* un score de confiance

Dans notre projet Alyra, après benchmark (WER, CER, temps d'exécution), nous avons retenu **Tesseract** comme OCR de référence.

Cette décision est fondée sur une expérimentation, même si elle a été réalisée sur un petit échantillon.

---

## 3. Ce que produit réellement Tesseract

Tesseract ne renvoie pas un texte continu.

Il renvoie une liste de mots.

Pour chaque mot, nous obtenons :

```python
{
    "text": "...",
    "confidence": ...,
    "bbox": [...]
}
```

Exemple :

```python
{
    "text": "CHEVAL",
    "confidence": 0.96,
    "bbox": [286,415,388,433]
}
```

Nous avons compris que LayoutLMv3 ne travaille pas avec des phrases mais avec des **tokens**.

---

## 4. Les Bounding Boxes

Chaque mot possède une boîte englobante.

Cette boîte indique où se trouve le mot sur la page.

Format :

```text
[x_min,
 y_min,
 x_max,
 y_max]
```

Les bounding boxes permettent au modèle de comprendre la disposition spatiale du document.

---

## 5. Pourquoi normaliser les coordonnées

Les coordonnées fournies par Tesseract sont exprimées en pixels.

Ces valeurs dépendent de la résolution de l'image.

Deux scans du même document peuvent produire des coordonnées totalement différentes.

Pour rendre le modèle indépendant de la résolution, LayoutLM normalise les coordonnées dans un espace commun :

```text
0
│
│
1000
```

Ainsi :

* un document de 1000 px
* un document de 3000 px

produisent les mêmes coordonnées normalisées.

Le choix de 1000 est une convention utilisée par la famille LayoutLM.

---

## 6. Le rôle de l'image

Nous avons compris que LayoutLMv3 exploite trois types d'information :

### Le texte

Exemple :

```
TOTAL
```

---

### Les positions

```
(820, 1500)
```

---

### L'image

L'image apporte des informations impossibles à déduire uniquement des tokens :

* tableaux
* bordures
* logos
* alignements
* taille des polices
* gras
* éléments graphiques

L'image fournit donc un **contexte visuel**.

---

## 7. Visualisation des bounding boxes

Nous avons dessiné les rectangles directement sur le devis.

Cette étape a permis de comprendre que LayoutLMv3 ne "voit" plus un devis mais une collection de mots géolocalisés.

Exemple :

```
Offre

de

prix

002.143/25

CHEVAL

...
```

Chaque mot est associé à une position.

---

## 8. Le véritable rôle des annotations

Avant cette séance, nous pensions que l'annotation consistait à dessiner des rectangles.

En réalité :

Les rectangles existent déjà.

Ils sont produits par l'OCR.

L'annotation consiste uniquement à répondre à la question :

> **Que représente ce token ?**

Exemple :

Avant :

```python
{
    "text":"CHEVAL",
    "bbox":[...]
}
```

Après annotation :

```python
{
    "text":"CHEVAL",
    "bbox":[...],
    "label":"CUSTOMER"
}
```

L'annotation est donc une **classification de tokens**.

---

## 9. Ce que reçoit réellement LayoutLMv3

Pendant l'entraînement, le modèle reçoit :

```text
Image

+

Tokens

+

Bounding Boxes

+

Labels
```

Pendant l'inférence :

```text
Image

+

Tokens

+

Bounding Boxes
```

Le modèle doit alors prédire les labels.

---

# Notebook 03 — Préparation de Label Studio

## Objectif

Comprendre comment passer de la sortie de Tesseract à un format exploitable par Label Studio.

Nous avons volontairement choisi de construire ce pipeline nous-mêmes afin de comprendre chaque transformation.

---

## Étapes réalisées

### Conversion du PDF en image

avec PyMuPDF.

---

### OCR

avec Tesseract.

---

### Construction de la liste :

```python
words = [
    {
        "text": ...,
        "confidence": ...,
        "bbox_pixels": ...
    }
]
```

---

### Conversion des bounding boxes

Pixels

↓

Pourcentages

car Label Studio travaille avec des coordonnées relatives.

---

### Construction des régions

Pour chaque mot :

* rectangle
* transcription

Chaque paire partage le même identifiant.

---

### Génération du JSON

Nous avons construit manuellement le JSON attendu par Label Studio.

Cela nous a permis de comprendre :

* la structure du fichier
* le rôle de chaque champ
* le lien entre OCR et annotation

---

# Pipeline obtenu

À la fin de cette séance, nous maîtrisons le pipeline suivant :

```text
PDF

↓

PyMuPDF

↓

Image

↓

Tesseract

↓

Tokens

↓

Bounding Boxes

↓

Conversion Label Studio

↓

JSON importable
```

---

# Ce qu'il reste à faire

## 1

Configurer Label Studio.

Comprendre la configuration XML.

---

## 2

Importer notre JSON.

---

## 3

Réaliser les annotations.

---

## 4

Exporter les annotations.

---

## 5

Convertir les annotations au format attendu par LayoutLMv3.

---

## 6

Construire le dataset final.

---

# Ce qu'il faut retenir pour la soutenance

Les principaux messages à retenir sont :

1. **Le choix de l'OCR est fondé sur une expérimentation** (WER, CER, temps d'exécution), même si l'échantillon est limité.

2. **LayoutLMv3 ne reçoit jamais un PDF directement.** Le PDF est converti en image, puis enrichi par les résultats de l'OCR.

3. **L'OCR fournit les tokens et leurs positions.** Les annotations n'ont pas pour rôle de redétecter les mots, mais de leur attribuer un label métier.

4. **L'annotation est une tâche de classification de tokens**, pas de détection d'objets.

5. **L'ensemble du pipeline est désormais compris et maîtrisé**, de la lecture du PDF jusqu'à la génération d'un JSON importable dans Label Studio. C'était l'objectif principal de ces deux notebooks avant d'aborder le fine-tuning.
